# Revision de los datos de restauracion

Primero mirar, despues decidir, y solo al final escribir un script y un `gold`. Este cuaderno es la
fase de mirar: **no produce ningun CSV**.

Tres fuentes sobre la mesa:

| Fuente | Cobertura | Fecha |
|---|---|---|
| OSM | toda la provincia | 2026 |
| Censo comercial del Ajuntament | solo Barcelona ciudad | octubre 2024 |
| Terrazas del Ajuntament | solo Barcelona ciudad | 1 de julio de 2026 |

Las preguntas, en orden:

1. Cuantos locales hay en cada fuente y por que no coinciden
2. Que campos trae cada una y cuales estan vacios
3. Si el cruce de terrazas es fiable o no


In [1]:
# 1. Carga
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw" / "restauracion_hoteles_provincia"

osm = pd.read_csv(RAIZ / "data" / "bronze" / "restauracion_con_municipio.csv", low_memory=False)
censo = pd.read_csv(RAW / "bcn_cens_comercial_restauracion_2024.csv", low_memory=False)
terrazas = pd.read_csv(RAW / "bcn_terrasses_restauracio_2026.csv", low_memory=False)

print(f"OSM      {osm.shape[0]:>7,} filas x {osm.shape[1]:>2} columnas   (provincia)")
print(f"censo    {censo.shape[0]:>7,} filas x {censo.shape[1]:>2} columnas   (ciudad)")
print(f"terrazas {terrazas.shape[0]:>7,} filas x {terrazas.shape[1]:>2} columnas   (ciudad)")

OSM       16,801 filas x 16 columnas   (provincia)
censo     10,864 filas x 49 columnas   (ciudad)
terrazas   6,942 filas x 20 columnas   (ciudad)


## 1. Cuantos locales hay en Barcelona ciudad

Las dos fuentes que cubren la ciudad no dicen lo mismo, y la diferencia no es pequena.

In [2]:
# 2. OSM frente al censo, en la ciudad
osm_bcn = osm[osm["municipio_poligono"] == "Barcelona"]
# El censo mete restauracion y alojamiento en el mismo grupo de actividad. Los de alojamiento
# cuentan en el otro lado del analisis, con los hoteles.
es_alojamiento = censo["Nom_Activitat"].str.contains("allotjament", case=False, na=False)
censo_rest = censo[~es_alojamiento]

print("=== LOCALES DE RESTAURACION EN BARCELONA CIUDAD ===")
print(f"  OSM (cartografia voluntaria) : {len(osm_bcn):>7,}")
print(f"  Censo municipal 2024         : {len(censo_rest):>7,}"
      f"   (mas {int(es_alojamiento.sum())} de alojamiento)")
print(f"  Diferencia                   : {len(censo_rest) - len(osm_bcn):>7,}"
      f"   ({1 - len(osm_bcn) / len(censo_rest):.0%} menos en OSM)")
print()
print("Referencias externas, para saber cual de las dos es creible:")
print("  Ajuntament, guia de restauracion 2025 : 'mas de diez mil'")
print("  Censo municipal de 2017               : 9.359 bares y restaurantes")

=== LOCALES DE RESTAURACION EN BARCELONA CIUDAD ===
  OSM (cartografia voluntaria) :   7,430
  Censo municipal 2024         :  10,100   (mas 764 de alojamiento)
  Diferencia                   :   2,670   (26% menos en OSM)

Referencias externas, para saber cual de las dos es creible:
  Ajuntament, guia de restauracion 2025 : 'mas de diez mil'
  Censo municipal de 2017               : 9.359 bares y restaurantes


## 2. Que trae cada fuente

Interesa sobre todo **que esta vacio**, porque es lo que decide para que sirve cada una.

In [3]:
# 3. Cobertura de campos
def cobertura(d, campos):
    return pd.DataFrame({
        "informados": [int(d[c].notna().sum()) for c in campos],
        "%": [round(d[c].notna().mean() * 100, 1) for c in campos],
    }, index=campos)


print("=== OSM, provincia entera ===")
display(cobertura(osm, ["nombre_comercial", "tipo_local", "tipo_cocina", "calle", "numero",
                        "codigo_postal", "telefono", "sitio_web", "latitud", "barrio"]))

print("=== CENSO, ciudad ===")
display(cobertura(censo_rest, ["Nom_Local", "Nom_Activitat", "Nom_Barri", "Nom_Via",
                               "Num_Policia_Inicial", "Latitud"]))

print("=== TERRAZAS, ciudad ===")
display(cobertura(terrazas, ["EMPLACAMENT", "NOM_BARRI", "TAULES", "CADIRES",
                             "SUPERFICIE_OCUPADA", "LATITUD", "VIGENCIA"]))

=== OSM, provincia entera ===


,informados,%
nombre_comercial,15570,92.7
tipo_local,16801,100.0
tipo_cocina,5527,32.9
calle,7631,45.4
numero,6757,40.2
codigo_postal,4910,29.2
telefono,3591,21.4
sitio_web,2660,15.8
latitud,16801,100.0
barrio,7430,44.2


=== CENSO, ciudad ===


,informados,%
Nom_Local,10100,100.0
Nom_Activitat,10100,100.0
Nom_Barri,10100,100.0
Nom_Via,10100,100.0
Num_Policia_Inicial,10100,100.0
Latitud,10100,100.0


=== TERRAZAS, ciudad ===


,informados,%
EMPLACAMENT,6942,100.0
NOM_BARRI,6942,100.0
TAULES,6942,100.0
CADIRES,6942,100.0
SUPERFICIE_OCUPADA,6942,100.0
LATITUD,6942,100.0
VIGENCIA,6942,100.0


In [4]:
# 4. Los dos huecos de OSM, en numeros
print("=== TIPO DE COCINA ===")
print(f"vacio en {int(osm['tipo_cocina'].isna().sum()):,} de {len(osm):,} "
      f"({osm['tipo_cocina'].isna().mean():.0%}) en la provincia")
print()
print("Lo mas fino que ofrece el censo, para ver si puede rellenarlo:")
for k, v in censo_rest["Nom_Activitat"].value_counts().head(6).items():
    print(f"  {v:>5,}  {str(k)[:60]}")
print()
print("  -> son tipos de LOCAL, no de cocina. Ninguna fuente distingue japones de italiano.")
print()
print("=== NOMBRE COMERCIAL ===")
sin_nombre = osm_bcn["nombre_comercial"].isna()
print(f"vacio en {int(sin_nombre.sum())} de {len(osm_bcn):,} en la ciudad ({sin_nombre.mean():.1%})")
print(f"de esos, con calle informada: {int(osm_bcn.loc[sin_nombre, 'calle'].notna().sum())}")
print("  -> sin nombre y casi sin direccion: solo se podrian emparejar por coordenada.")

=== TIPO DE COCINA ===
vacio en 11,274 de 16,801 (67%) en la provincia

Lo mas fino que ofrece el censo, para ver si puede rellenarlo:
  4,430  Restaurants
  4,273  Bars   / CIBERCAFÈ
    778  Serveis de menjar take away MENJAR RÀPID
    387  Bars especials amb actuació / Bars musicals / Discoteques /P
    148  Xocolateries / Geladeries / Degustació
     74  serveis de menjar i begudes

  -> son tipos de LOCAL, no de cocina. Ninguna fuente distingue japones de italiano.

=== NOMBRE COMERCIAL ===
vacio en 351 de 7,430 en la ciudad (4.7%)
de esos, con calle informada: 68
  -> sin nombre y casi sin direccion: solo se podrian emparejar por coordenada.


## 3. Que aportan las terrazas

Es la unica medida de **capacidad** de toda la capa: el resto cuenta locales, no plazas.

In [5]:
# 5. Las terrazas por dentro
print("=== TERRAZAS, volcado del 1 de julio de 2026 ===")
print(f"  licencias   : {len(terrazas):>7,}")
print(f"  mesas       : {int(terrazas['TAULES'].sum()):>7,}")
print(f"  sillas      : {int(terrazas['CADIRES'].sum()):>7,}")
print(f"  superficie  : {terrazas['SUPERFICIE_OCUPADA'].sum():>7,.0f} m2 de via publica")
print(f"  barrios     : {terrazas['NOM_BARRI'].nunique():>7}")
print()
print("Referencia externa: 6.899 licencias en 2025, +220 concedidas ese ano.")
print()
for c in ["VIGENCIA", "OCUPACIO"]:
    print(f"{c}: {terrazas[c].value_counts().to_dict()}")
print()
print(f"emplazamientos distintos: {terrazas['EMPLACAMENT'].nunique():,} "
      f"para {len(terrazas):,} licencias")
print("  -> hay locales con mas de una licencia: licencias no es lo mismo que locales con terraza")
print()
print("NO trae nombre de local ni identificador del censo: el vinculo hay que construirlo.")

=== TERRAZAS, volcado del 1 de julio de 2026 ===
  licencias   :   6,942
  mesas       :  33,166
  sillas      : 127,482
  superficie  :  83,745 m2 de via publica
  barrios     :      70

Referencia externa: 6.899 licencias en 2025, +220 concedidas ese ano.

VIGENCIA: {'Anual': 6851, 'Temporada': 91}
OCUPACIO: {'Terrasses en Via Pública': 6762, "Terrasses en Espai Privat d'Ús Públic": 180}

emplazamientos distintos: 6,227 para 6,942 licencias
  -> hay locales con mas de una licencia: licencias no es lo mismo que locales con terraza

NO trae nombre de local ni identificador del censo: el vinculo hay que construirlo.


## 4. El cruce de terrazas: se puede o no

Aqui esta la decision. Se prueban dos vias y se miden **contra la proporcion real**, que se conoce:
las licencias menos los emplazamientos repetidos, sobre los locales del censo.

In [6]:
# 6. Calibracion del cruce
import re
import unicodedata

import numpy as np
from scipy.spatial import cKDTree

PREFIJOS = (r"^(c|carrer|av|avinguda|avgda|pg|passeig|pl|placa|rbla|rambla|ctra|via|gv|trav|"
            r"travessera|ronda|bxda|baixada|ptge|passatge)\b\.?\s*")


def normalizar_via(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", re.sub(PREFIJOS, "", s))).strip()


def partir(emplazamiento):
    """`AV. GAUDI, 46` -> (`gaudi`, 46). Los tramos `419-425` se quedan con el inicial."""
    m = re.match(r"^(.*?),\s*(\d+)", str(emplazamiento))
    return (normalizar_via(m.group(1)), float(m.group(2))) if m else (normalizar_via(emplazamiento),
                                                                      np.nan)


LAT0 = 41.39   # a esta latitud un grado de longitud mide 83 km y uno de latitud 111


def proyectar(lat, lon):
    return np.c_[np.asarray(lon, float) * 111320 * np.cos(np.radians(LAT0)),
                 np.asarray(lat, float) * 110570]


t = terrazas.copy()
t[["via", "num"]] = pd.DataFrame([partir(x) for x in t["EMPLACAMENT"]], index=t.index)
c = censo_rest.copy().reset_index(drop=True)
c["via"] = c["Nom_Via"].map(normalizar_via)
c["num"] = pd.to_numeric(c["Num_Policia_Inicial"], errors="coerce")

direcciones = set(zip(t["via"], t["num"]))
por_direccion = pd.Series([(v, n) in direcciones for v, n in zip(c["via"], c["num"])], index=c.index)
arbol = cKDTree(proyectar(t["LATITUD"], t["LONGITUD"]))
por_distancia = pd.Series([len(v) > 0 for v in
                           arbol.query_ball_point(proyectar(c["Latitud"], c["Longitud"]), r=10)],
                          index=c.index)
union = por_direccion | por_distancia

repetidos = len(t) - t["EMPLACAMENT"].nunique()
esperado = (len(t) - repetidos) / len(c)

print("=== CUANTOS LOCALES QUEDARIAN MARCADOS CON TERRAZA ===")
for nombre, marca in [("solo por direccion", por_direccion),
                      ("solo por distancia (10 m)", por_distancia),
                      ("union de las dos", union)]:
    print(f"  {nombre:<26} {int(marca.sum()):>6,}  ({marca.mean():.0%})")
print()
print(f"  proporcion REAL esperada   {int(esperado * len(c)):>6,}  ({esperado:.0%})")
print()
print(f"  los dos metodos coinciden en el {(por_direccion == por_distancia).mean():.0%} de los locales")
fallo = esperado - union.mean()
print(f"  se quedan sin marcar {fallo * len(c):,.0f} locales que SI tienen terraza: "
      f"{fallo / esperado:.0%} de los verdaderos")

=== CUANTOS LOCALES QUEDARIAN MARCADOS CON TERRAZA ===
  solo por direccion          4,981  (49%)
  solo por distancia (10 m)   4,728  (47%)
  union de las dos            5,661  (56%)

  proporcion REAL esperada    6,227  (62%)

  los dos metodos coinciden en el 84% de los locales
  se quedan sin marcar 566 locales que SI tienen terraza: 9% de los verdaderos


In [7]:
# 7. Los falsos positivos: portales que marcan a quien no toca
tam = c.groupby(["via", "num"])["ID_Global"].transform("size")
marcados = c[union]

print("=== UNA LICENCIA EN UN PORTAL MARCA A TODOS LOS LOCALES DEL PORTAL ===")
peor = marcados.groupby(["via", "num"]).size().sort_values(ascending=False).head(5)
for (via, num), n in peor.items():
    print(f"  {via} {num:.0f}  ->  {n} locales marcados")
print()
print(f"marcados en direcciones con mas de 3 locales: "
      f"{int((union & tam.gt(3)).sum()):,} de {int(union.sum()):,}")
print()
interior = c["SN_CComercial"].eq("Si") | c["SN_Mercat"].eq("Si") | c["SN_Galeria"].eq("Si")
print(f"marcados dentro de centro comercial, mercado o galeria: {int((union & interior).sum()):,}")
print("  (su portal puede tener licencia de terraza, pero no es suya)")

=== UNA LICENCIA EN UN PORTAL MARCA A TODOS LOS LOCALES DEL PORTAL ===
  potosi 2  ->  51 locales marcados
  diagonal 208  ->  30 locales marcados
  diagonal 557  ->  24 locales marcados
  g v corts catalanes 373  ->  22 locales marcados
  castillejos 158  ->  11 locales marcados

marcados en direcciones con mas de 3 locales: 211 de 5,661

marcados dentro de centro comercial, mercado o galeria: 170
  (su portal puede tener licencia de terraza, pero no es suya)


## 5. Que queda decidido y que no

**Descartado, sin vuelta:**

- **El tipo de cocina no esta en ninguna fuente.** No hay nada que decidir: no se puede rellenar.
- **Los nombres se podrian copiar del censo, pero uno de cada cuatro seria falso.** Calibrado contra
  los locales de OSM que si tienen nombre, a menos de 8 m y con candidato unico, el 25% resulta ser
  un negocio distinto --el censo es de 2024 y OSM de 2026--. Y en los que no tienen nombre no hay
  forma de saber cual es cual, porque no hay con que contrastar.

**Pendiente de decidir, con los numeros de arriba delante:**

1. **Que fuente usar dentro de la ciudad.** El recuento de OSM no es compatible con ninguna cifra
   oficial.
2. **Si usar el cruce de terrazas.** Las celdas 6 y 7 dicen cuanto se pierde y cuanto se ensucia.

Cuando esas dos esten decididas, y solo entonces, se escribe el script y se genera el `gold`.
